# Disease ↔ Phenotype Relation-Wise Merge

Merges Disease–Phenotype triples from Monarch and CrossBAR; resolves disease names
via DO/MESH; deduplicates by `(head, relation, tail)`; and saves the result.

## 0. Configuration

In [1]:
import pandas as pd
import numpy as np

BASE_DIR   = '/storage/Arushi/090526_EvoAge/kg_formation/'
PROC_DIR   = BASE_DIR + 'processed_data/'
DB_DIR     = BASE_DIR + 'data_collection/databases_for_mapping/'

OUT_PATH   = BASE_DIR + 'processed_data_relation_wise_merge/generalised/DISEASE_PATHWAY/ALL_DISEASE_PATHWAY.csv'

REQUIRED_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source',
    'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', 'kg_type', 'species'
]

## 1. Build Disease Lookup Dictionaries

In [2]:
# Disease Ontology
DO_All_File = pd.read_csv(DB_DIR + 'DO/All_DO_ref_files.csv')
DOID_disname_dict      = dict(zip(DO_All_File['id'],    DO_All_File['label']))
DOID_disAltID_dict     = dict(zip(DO_All_File['id'],    DO_All_File['xrefs']))
DOID_disname_DOID_dict = dict(zip(DO_All_File['label'], DO_All_File['id']))
print(f"DO dict: {len(DOID_disname_dict):,} entries")

DO dict: 11,804 entries


In [3]:
# MedGen: source_id → preferred name; MONDO → MESH CUI
MedGen_CUID_Source_ID = pd.read_csv(DB_DIR + 'MESH/MeSH/MedGenIDMappings.txt', sep='\t')
MedGen_CUID_Source_ID = MedGen_CUID_Source_ID.drop_duplicates(subset=['source_id', 'source'], keep='first')
MedGen_MeshID_name_dict = dict(zip(MedGen_CUID_Source_ID['source_id'], MedGen_CUID_Source_ID['pref_name']))

MONDO_2_MESH = MedGen_CUID_Source_ID[MedGen_CUID_Source_ID['source_id'].str.contains('MONDO', na=False)]
MONDO_2_MESH_dict    = dict(zip(MONDO_2_MESH['source_id'],     MONDO_2_MESH['#CUI_or_CN_id']))
MONDOMESH_2_des_dict = dict(zip(MONDO_2_MESH['#CUI_or_CN_id'], MONDO_2_MESH['pref_name']))
print(f"MONDO→MESH: {len(MONDO_2_MESH_dict):,} entries")

MONDO→MESH: 22,703 entries


In [4]:
# MESH combined and supplementary: ID → Name
Mesh2DOID = pd.read_csv(DB_DIR + 'DO/HumanDiseaseOntology-main/DOreports/MESHinDO.tsv', sep='\t')
Mesh2DOID.rename(columns={'ID': 'id', 'LABEL': 'label'}, inplace=True)
Mesh2DOID['xrefs'] = Mesh2DOID['xrefs'].str.split(', ')
Mesh2DOID = Mesh2DOID.explode('xrefs')
Mesh2DOID_dict = dict(zip(Mesh2DOID['xrefs'], Mesh2DOID['id']))

MESH = pd.read_csv(DB_DIR + 'MESH/MESH_Combined.csv')
MESH_dict = dict(zip(MESH['ID'], MESH['Name']))

Mesh_supp = pd.read_csv(DB_DIR + 'MESH/Mesh_Supplementary_Info.csv')
Mesh_supp_dict = dict(zip(Mesh_supp['ID'], Mesh_supp['Name']))

print(f"MESH dict: {len(MESH_dict):,} | MESH supp: {len(Mesh_supp_dict):,}")

MESH dict: 38,520 | MESH supp: 324,150


In [30]:
Reactome = pd.read_csv(DB_DIR + 'reactome/ReactomePathways.txt', sep = '\t',header = None)
Reactome
Reactome_dict = dict(zip(Reactome[0], Reactome[1]))
Reactome_dict

{'R-BTA-73843': '5-Phosphoribose 1-diphosphate biosynthesis',
 'R-BTA-1971475': 'A tetrasaccharide linker sequence is required for GAG synthesis',
 'R-BTA-1369062': 'ABC transporters in lipid homeostasis',
 'R-BTA-382556': 'ABC-family proteins mediated transport',
 'R-BTA-9033807': 'ABO blood group biosynthesis',
 'R-BTA-418592': 'ADP signalling through P2Y purinoceptor 1',
 'R-BTA-392170': 'ADP signalling through P2Y purinoceptor 12',
 'R-BTA-211163': 'AKT-mediated inactivation of FOXO1A',
 'R-BTA-163680': 'AMPK inhibits chREBP transcriptional activation activity',
 'R-BTA-179409': 'APC-Cdc20 mediated degradation of Nek2A',
 'R-BTA-174143': 'APC/C-mediated degradation of cell cycle proteins',
 'R-BTA-174048': 'APC/C:Cdc20 mediated degradation of Cyclin B',
 'R-BTA-174154': 'APC/C:Cdc20 mediated degradation of Securin',
 'R-BTA-176409': 'APC/C:Cdc20 mediated degradation of mitotic proteins',
 'R-BTA-174178': 'APC/C:Cdh1 mediated degradation of Cdc20 and other APC/C:Cdh1 targeted protei

## 2. Helper — Resolve Disease Head Node

In [5]:
def resolve_disease_node(df, node):
    """Map MONDO→MESH, derive detail_name, assign id_is for head or tail."""
    df[node] = df[node].map(MONDO_2_MESH_dict).fillna(df[node])
    df = df[~df[node].astype(str).str.startswith('MONDO')].copy()
    df[f'{node}_detail_name'] = df[node].apply(
        lambda x: DOID_disname_dict.get(x)
                  if isinstance(x, str) and x.startswith('DOID')
                  else (MESH_dict.get(x) or Mesh_supp_dict.get(x) or MONDOMESH_2_des_dict.get(x))
    )
    df[f'{node}_id_is'] = np.where(
        df[node].isna(), None,
        np.where(df[node].astype(str).str.startswith('DOID'), 'DOID', 'MESH')
    )
    return df

## 3. Load KG Sources

### 3a. iBKH

In [15]:
iBKH_Disease_Pathway = pd.read_csv(PROC_DIR + 'iBKH/Disease_Pathway_ibkh.csv')
iBKH_Disease_Pathway.columns = iBKH_Disease_Pathway.columns.str.lower()

iBKH_Disease_Pathway['kg_type'] = 'Generalised'
iBKH_Disease_Pathway['kg_source'] = 'iBKH'
iBKH_Disease_Pathway['species'] = 'Homo species'

iBKH_Disease_Pathway.head(2)

,head,relation,tail,head_type,tail_type,kg_source,head_detail_name,tail_detail_name,head_id_is,tail_id_is,tail_detail_name.1,kg_type,species
0,DOID:2747,Disease_Pathway,KEGG:map00010,Disease,Pathway,iBKH,glycogen storage disease,Glycolysis / Gluconeogenesis,DOID,KEGG_ID,Glycolysis / Gluconeogenesis,Generalised,Homo species
1,DOID:9869,Disease_Pathway,KEGG:map00010,Disease,Pathway,iBKH,hereditary fructose intolerance syndrome,Glycolysis / Gluconeogenesis,DOID,KEGG_ID,Glycolysis / Gluconeogenesis,Generalised,Homo species


## 4. Check and Fix Duplicate Columns

In [16]:
SOURCE_DFS = [
    ('iBKH_Disease_Pathway',  iBKH_Disease_Pathway),
    
]

for name, df in SOURCE_DFS:
    dupes = df.columns[df.columns.duplicated(keep=False)].unique().tolist()
    if dupes:
        print(f"\n[{name}] duplicate columns: {dupes}")
        for col in dupes:
            for i, (_, s) in enumerate(df.loc[:, df.columns == col].items()):
                print(f"  '{col}' occurrence {i} → sample: {s.dropna().head(3).tolist()}")
    else:
        print(f"[{name}] ✓ no duplicates")

[iBKH_Disease_Pathway] ✓ no duplicates


## 5. Align Schemas and Concatenate

In [33]:
CONCAT_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source', 'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name','kg_type', 'species'
]

df_list = []
for name, df in SOURCE_DFS:
    tmp = df.loc[:, ~df.columns.duplicated()].copy()
    for col in CONCAT_COLS:
        if col not in tmp.columns:
            tmp[col] = pd.NA
    df_list.append(tmp[CONCAT_COLS])

final_df = pd.concat(df_list, ignore_index=True)
print(f"Combined: {len(final_df):,} rows")
final_df.head(2)

Combined: 1,039 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,DOID:2747,Disease_Pathway,KEGG:map00010,Disease,<NA>,Pathway,iBKH,DOID,KEGG_ID,glycogen storage disease,Glycolysis / Gluconeogenesis,Generalised,Homo species
1,DOID:9869,Disease_Pathway,KEGG:map00010,Disease,<NA>,Pathway,iBKH,DOID,KEGG_ID,hereditary fructose intolerance syndrome,Glycolysis / Gluconeogenesis,Generalised,Homo species


In [34]:
for col in ['relation', 'head_type', 'tail_type', 'relation_type', 'kg_source', 'head_id_is', 'tail_id_is', 'kg_type','species']:
    print(f"{col:20s}: {set(final_df[col])}")

relation            : {'Disease_Pathway'}
head_type           : {'Disease'}
tail_type           : {'Pathway'}
relation_type       : {<NA>}
kg_source           : {'iBKH'}
head_id_is          : {'DOID'}
tail_id_is          : {'Reactome_ID', 'KEGG_ID'}
kg_type             : {'Generalised'}
species             : {'Homo species'}


## 6. Standardise Fixed-Value Columns

In [35]:
print("Unique relation:",   set(final_df['relation']))
print("Unique head_id_is:", set(final_df['head_id_is']))
print("Unique kg_source:",  set(final_df['kg_source']))

Unique relation: {'Disease_Pathway'}
Unique head_id_is: {'DOID'}
Unique kg_source: {'iBKH'}


## 7. Deduplicate and Add Schema Columns

In [36]:
GROUP_COLS = ['head', 'relation', 'tail']

def merge_sources(x):
    return '::'.join(sorted(set(x.dropna())))

final_df_dedup = final_df.groupby(GROUP_COLS, as_index=False).agg({
    'head_type':        'first',
    'relation_type':    'first',
    'tail_type':        'first',
    'kg_source':         merge_sources,
    'head_id_is':       'first',
    'tail_id_is':       'first',
    'head_detail_name': 'first',
    'tail_detail_name': 'first',
    'kg_type': merge_sources,
    'species': 'first',
})
final_df_dedup['head'] = final_df_dedup['head'].astype(str)
print(f"After deduplication: {len(final_df_dedup):,} rows")

After deduplication: 1,036 rows


## 8. QC — NaN Check

In [37]:
def nan_summary(df):
    true_nan   = df.isna().sum()
    string_nan = df.apply(lambda c: c.astype(str).str.upper().eq('NAN').sum())
    return pd.DataFrame({
        'NaN_count':          true_nan,
        "'NAN'_string_count": string_nan,
        'Total_NaN_like':     true_nan + string_nan
    })

nan_summary(final_df_dedup)

,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,1036,0,1036
tail_type,0,0,0
kg_source,0,0,0
head_id_is,0,0,0
tail_id_is,0,0,0
head_detail_name,0,0,0


In [38]:
final_df_dedup = final_df_dedup[~final_df_dedup['tail_detail_name'].isna()]
final_df_dedup

,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,DOID:0050328,Disease_Pathway,KEGG:map04919,Disease,None,Pathway,iBKH,DOID,KEGG_ID,congenital hypothyroidism,Thyroid hormone signaling pathway,Generalised,Homo species
2,DOID:0050331,Disease_Pathway,KEGG:map04144,Disease,None,Pathway,iBKH,DOID,KEGG_ID,lacrimoauriculodentodigital syndrome 1,Endocytosis,Generalised,Homo species
3,DOID:0050331,Disease_Pathway,KEGG:map04810,Disease,None,Pathway,iBKH,DOID,KEGG_ID,lacrimoauriculodentodigital syndrome 1,Regulation of actin cytoskeleton,Generalised,Homo species
5,DOID:0050335,Disease_Pathway,KEGG:map04744,Disease,None,Pathway,iBKH,DOID,KEGG_ID,bradyopsia,Phototransduction,Generalised,Homo species
6,DOID:0050424,Disease_Pathway,KEGG:map03410,Disease,None,Pathway,iBKH,DOID,KEGG_ID,familial adenomatous polyposis,Base excision repair,Generalised,Homo species
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1030,DOID:9955,Disease_Pathway,KEGG:map04540,Disease,None,Pathway,iBKH,DOID,KEGG_ID,hypoplastic left heart syndrome,Gap junction,Generalised,Homo species
1031,DOID:9965,Disease_Pathway,KEGG:map05145,Disease,None,Pathway,iBKH,DOID,KEGG_ID,toxoplasmosis,Toxoplasmosis,Generalised,Homo species
1032,DOID:9970,Disease_Pathway,KEGG:map03320,Disease,None,Pathway,iBKH,DOID,KEGG_ID,obesity,PPAR signaling pathway,Generalised,Homo species
1033,DOID:9970,Disease_Pathway,KEGG:map04080,Disease,None,Pathway,iBKH,DOID,KEGG_ID,obesity,Neuroactive ligand-receptor interaction,Generalised,Homo species


## 9. Save Output

In [39]:
import os
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
final_df_dedup.to_csv(OUT_PATH, index=False)
print(f"Saved {len(final_df_dedup):,} rows → {OUT_PATH}")

Saved 866 rows → /storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/DISEASE_PATHWAY/ALL_DISEASE_PATHWAY.csv


# Final Mapping 

In [2]:
# =========================================================
# LOAD DISEASE MAPPING FILE
# =========================================================

DB_DIR = BASE_DIR + 'data_collection/databases_for_mapping/'

final_file_CH_D = pd.read_csv(
    DB_DIR + 'DO/ultimate_DISEASE_dictionary.csv'
)

# =========================================================
# PRIORITIZE DOID IDs
# =========================================================

# rows having DOID
doid_rows = final_file_CH_D[
    final_file_CH_D['entity_id']
    .str.startswith('DOID:', na=False)
]

# entity names that have at least one DOID
names_with_doid = set(
    doid_rows['entity_name']
    .str.lower()
    .str.strip()
)

# keep:
# 1. all DOID rows
# 2. non-DOID rows only if no DOID exists

final_diseaseid_df = final_file_CH_D[

    final_file_CH_D['entity_id']
    .str.startswith('DOID:', na=False)

    |

    ~final_file_CH_D['entity_name']
    .str.lower()
    .str.strip()
    .isin(names_with_doid)
]

final_diseaseid_df = (
    final_diseaseid_df
    .reset_index(drop=True)
)

# =========================================================
# CREATE DISEASE NAME -> ID DICTIONARY
# =========================================================

disease_dict = dict(

    zip(

        final_diseaseid_df['entity_name']
        .str.lower()
        .str.strip(),

        final_diseaseid_df['entity_id']
    )
)

# =========================================================
# UNIVERSAL ENTITY MAPPING FUNCTION
# =========================================================

def map_entity_ids(
    df,
    mapping_dict,
    side='tail'
):
    """
    Universal KG entity mapper.

    Parameters
    ----------
    df : pd.DataFrame

    mapping_dict : dict
        entity_name -> entity_id

    side : str
        'head' or 'tail'

    Returns
    -------
    pd.DataFrame
    """

    # -----------------------------------------
    # dynamic column names
    # -----------------------------------------

    detail_col = f'{side}_detail_name'

    id_col = side

    id_is_col = f'{side}_id_is'

    # -----------------------------------------
    # map IDs
    # -----------------------------------------

    df[id_col] = (df[detail_col].astype(str).str.lower().str.strip().map(mapping_dict))

    # -----------------------------------------
    # assign ID source
    # -----------------------------------------

    df[id_is_col] = None

    # DOID
    # DOID
    df.loc[
        df[id_col].str.startswith('DOID:', na=False),id_is_col] = 'DOID'
    
    # everything else -> MESH
    df.loc[~df[id_col].str.startswith('DOID:', na=False)&df[id_col].notna(),id_is_col] = 'MESH'

    return df

In [3]:
final_file = pd.read_csv(OUT_PATH)

final_file = map_entity_ids(df=final_file,mapping_dict=disease_dict,side='head')

final_file

,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species,head_species,tail_species
0,DOID:0050328,Disease_Pathway,KEGG:map04919,Disease,NaN,Pathway,iBKH,DOID,KEGG_ID,congenital hypothyroidism,Thyroid hormone signaling pathway,Generalised,Homo species,Homo sapiens,Homo sapiens
1,DOID:0050331,Disease_Pathway,KEGG:map04144,Disease,NaN,Pathway,iBKH,DOID,KEGG_ID,lacrimoauriculodentodigital syndrome 1,Endocytosis,Generalised,Homo species,Homo sapiens,Homo sapiens
2,DOID:0050331,Disease_Pathway,KEGG:map04810,Disease,NaN,Pathway,iBKH,DOID,KEGG_ID,lacrimoauriculodentodigital syndrome 1,Regulation of actin cytoskeleton,Generalised,Homo species,Homo sapiens,Homo sapiens
3,DOID:0050335,Disease_Pathway,KEGG:map04744,Disease,NaN,Pathway,iBKH,DOID,KEGG_ID,bradyopsia,Phototransduction,Generalised,Homo species,Homo sapiens,Homo sapiens
4,DOID:0050424,Disease_Pathway,KEGG:map03410,Disease,NaN,Pathway,iBKH,DOID,KEGG_ID,familial adenomatous polyposis,Base excision repair,Generalised,Homo species,Homo sapiens,Homo sapiens
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
861,DOID:9955,Disease_Pathway,KEGG:map04540,Disease,NaN,Pathway,iBKH,DOID,KEGG_ID,hypoplastic left heart syndrome,Gap junction,Generalised,Homo species,Homo sapiens,Homo sapiens
862,DOID:9965,Disease_Pathway,KEGG:map05145,Disease,NaN,Pathway,iBKH,DOID,KEGG_ID,toxoplasmosis,Toxoplasmosis,Generalised,Homo species,Homo sapiens,Homo sapiens
863,DOID:9970,Disease_Pathway,KEGG:map03320,Disease,NaN,Pathway,iBKH,DOID,KEGG_ID,obesity,PPAR signaling pathway,Generalised,Homo species,Homo sapiens,Homo sapiens
864,DOID:9970,Disease_Pathway,KEGG:map04080,Disease,NaN,Pathway,iBKH,DOID,KEGG_ID,obesity,Neuroactive ligand-receptor interaction,Generalised,Homo species,Homo sapiens,Homo sapiens


In [4]:
final_file[final_file['head'].isna()]


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species,head_species,tail_species


In [5]:
final_file.to_csv(OUT_PATH, index=False)
print(f"Saved {len(final_file):,} rows → {OUT_PATH}")

Saved 866 rows → /storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/DISEASE_PATHWAY/ALL_DISEASE_PATHWAY.csv
